In [2]:
import pandas as pd
from pathlib import Path

#set project root as .../recidivism-causal
project_root = Path("..").resolve()

#set causal pitfalls root
csv_path = project_root / "data" / "raw" / "NIJ_s_Recidivism_Chall.csv"

In [5]:
df = pd.read_csv(csv_path)
blank_cols = df.columns[df.isna().any()]
print(blank_cols)

Index(['Gang_Affiliated', 'Supervision_Risk_Score_First',
       'Supervision_Level_First', 'Prison_Offense', 'Avg_Days_per_DrugTest',
       'DrugTests_THC_Positive', 'DrugTests_Cocaine_Positive',
       'DrugTests_Meth_Positive', 'DrugTests_Other_Positive',
       'Percent_Days_Employed', 'Jobs_Per_Year'],
      dtype='object')


In [8]:
count_blank = df.isna().sum()
print(count_blank)
print((count_blank/25835)*100)

ID                                                      0
Gender                                                  0
Race                                                    0
Age_at_Release                                          0
Residence_PUMA                                          0
Gang_Affiliated                                      3167
Supervision_Risk_Score_First                          475
Supervision_Level_First                              1720
Education_Level                                         0
Dependents                                              0
Prison_Offense                                       3277
Prison_Years                                            0
Prior_Arrest_Episodes_Felony                            0
Prior_Arrest_Episodes_Misd                              0
Prior_Arrest_Episodes_Violent                           0
Prior_Arrest_Episodes_Property                          0
Prior_Arrest_Episodes_Drug                              0
Prior_Arrest_E

In [24]:
#conditional sampling imputation for gang affiliation
data = pd.read_csv(csv_path)
headers = list(data.columns)
print(headers)

from sdv.metadata import Metadata
from sdv.single_table import GaussianCopulaSynthesizer 
from sdv.sampling import DataFrameCondition

# build metadata
metadata = Metadata.detect_from_dataframe(
    data = data,
    table_name="NIJ"
)

synthesizer = GaussianCopulaSynthesizer(metadata=metadata)
synthesizer.fit(data)

#set conditioning columns for gang imputation
cond_cols_gang = [
    col for col in data.columns
    if col not in ["ID", "Gender", "Gang_Affiliated", 
                   "Recidivism_Within_3years", "Recidivism_Arrest_Year1",
                  "Recidivism_Arrest_Year2", "Recidivism_Arrest_Year3",
                  "Training_Sample"] #exclude these columns
]

#create mask for where rows where gang affiliation info is missing
missingness_mask= data["Gang_Affiliated"].isna()
known = data.loc[missingness_mask, cond_cols_gang]

#each row is one scenario to sample
conditions_df = DataFrameCondition(
    table_name="NIJ",
    dataframe=known
)

synthetic_conditional_gang=synthesizer.sample_from_conditions(
    conditions=[conditions_df]
)

#insert imputed values
gang_imputed=data.copy()
gang_imputed.loc[missingness_mask, "Gang_Affiliated"]=(
    synthetic_conditional_gang["Gang_Affiliated"].values
)

#add column to retain missingness information and save gang_imputed dataframe to csv
gang_imputation_flag = missingness_mask.astype(int)
gang_imputed["Gang_Affiliated_Missing"] = gang_imputation_flag
gang_imputed.to_csv("NIJ_gang_imputed.csv", index=False)

['ID', 'Gender', 'Race', 'Age_at_Release', 'Residence_PUMA', 'Gang_Affiliated', 'Supervision_Risk_Score_First', 'Supervision_Level_First', 'Education_Level', 'Dependents', 'Prison_Offense', 'Prison_Years', 'Prior_Arrest_Episodes_Felony', 'Prior_Arrest_Episodes_Misd', 'Prior_Arrest_Episodes_Violent', 'Prior_Arrest_Episodes_Property', 'Prior_Arrest_Episodes_Drug', 'Prior_Arrest_Episodes_PPViolationCharges', 'Prior_Arrest_Episodes_DVCharges', 'Prior_Arrest_Episodes_GunCharges', 'Prior_Conviction_Episodes_Felony', 'Prior_Conviction_Episodes_Misd', 'Prior_Conviction_Episodes_Viol', 'Prior_Conviction_Episodes_Prop', 'Prior_Conviction_Episodes_Drug', 'Prior_Conviction_Episodes_PPViolationCharges', 'Prior_Conviction_Episodes_DomesticViolenceCharges', 'Prior_Conviction_Episodes_GunCharges', 'Prior_Revocations_Parole', 'Prior_Revocations_Probation', 'Condition_MH_SA', 'Condition_Cog_Ed', 'Condition_Other', 'Violations_ElectronicMonitoring', 'Violations_Instruction', 'Violations_FailToReport', 'V

/dcs/23/u2200504/thesis/envthesis/lib/python3.13/site-packages/sdv/single_table/base.py:134: UserWarning:

We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.

Sampling conditions: 100%|██████████| 3167/3167 [19:23<00:00,  2.72it/s]


In [26]:
print(data["Gang_Affiliated"].isna().sum())
remain_blank=gang_imputed["Gang_Affiliated"].isna().sum()
print(remain_blank)
total = len(gang_imputed)
print(remain_blank/total*100) #~1.5% of data is still blank after imputation

3167
389
1.5057093090768339


In [57]:
#imputing drug related columns
cols_to_impute = [
    "Avg_Days_per_DrugTest",
    "DrugTests_THC_Positive",
    "DrugTests_Cocaine_Positive",
    "DrugTests_Meth_Positive",
    "DrugTests_Other_Positive",
]

cond_cols_drug = [
    col for col in data.columns
    if col not in ["ID", "Gang_Affiliated", 
                   "Recidivism_Within_3years", "Recidivism_Arrest_Year1",
                  "Recidivism_Arrest_Year2", "Recidivism_Arrest_Year3",
                  "Training_Sample"] + cols_to_impute #exclude these columns, dont use gang affiliated as we have fiddled with it
]

drug_imputed = gang_imputed.copy()

for col in cols_to_impute:
    # rows where this column is missing
    missing_mask = gang_imputed[col].isna()
    if missing_mask.sum() == 0:
        continue  # nothing to do for this column

    known_part = gang_imputed.loc[missing_mask, cond_cols_drug]

    conditions_df = DataFrameCondition(
        table_name="NIJ",
        dataframe=known_part
    )

    synthetic_conditional = synthesizer.sample_from_conditions(
        conditions=[conditions_df]
    )

    # copy back only this column’s synthetic values
    drug_imputed.loc[missing_mask, col] = synthetic_conditional[col].values
    #retain missingness information
    col_imputation_flag = missing_mask.astype(int)
    drug_imputed[f"{col}_Missing"] = col_imputation_flag

drug_imputed.to_csv("NIJ_drug_imputed.csv", index=False)
    

Sampling conditions: 100%|██████████| 5172/5172 [14:41<00:00,  5.87it/s]


In [59]:
#imputing prison_offense due to large proportion ~12.7% missing
original_cols = data.columns
prison_imputed = drug_imputed.copy()
target_col = "Prison_Offense"
exclude_from_conditions = [
    "ID",
    "Prison_Offense",               # target
    "Recidivism_Within_3years",
    "Recidivism_Arrest_Year1",
    "Recidivism_Arrest_Year2",
    "Recidivism_Arrest_Year3",
    "Gang_Affiliated"
] + cols_to_impute


cols_for_conditions = [
    c for c in original_cols
    if c not in exclude_from_conditions
]

missing_mask = drug_imputed[target_col].isna()

known_part = drug_imputed.loc[missing_mask, cols_for_conditions]

conditions_df = DataFrameCondition(
    table_name="NIJ",     # use your actual table name from metadata
    dataframe=known_part
)

synthetic_conditional = synthesizer.sample_from_conditions(
    conditions=[conditions_df]
)

prison_imputed.loc[missing_mask, target_col] = (
    synthetic_conditional[target_col].values
)

#add missing flags
prison_imputed["Prison_Offense_Missing"] = missing_mask.astype(int)

prison_imputed.to_csv(
    "NIJ_prison_offense_imputed.csv",
    index=False
)

Sampling conditions: 100%|██████████| 3277/3277 [09:18<00:00,  5.86it/s]


In [3]:
new_path = project_root / "data" / "raw" / "NIJ_prison_offense_imputed.csv"
imputed_df = pd.read_csv(new_path)

In [4]:
imputed_df.isna().sum()

ID                                    0
Gender                                0
Race                                  0
Age_at_Release                        0
Residence_PUMA                        0
                                     ..
DrugTests_THC_Positive_Missing        0
DrugTests_Cocaine_Positive_Missing    0
DrugTests_Meth_Positive_Missing       0
DrugTests_Other_Positive_Missing      0
Prison_Offense_Missing                0
Length: 61, dtype: int64

In [5]:
with pd.option_context("display.max_columns", None):
    print(imputed_df.iloc[[13]])    # 14th row, all columns shown

    ID Gender   Race Age_at_Release  Residence_PUMA Gang_Affiliated  \
13  14      M  BLACK          33-37               3            True   

    Supervision_Risk_Score_First Supervision_Level_First      Education_Level  \
13                           7.0                     NaN  High School Diploma   

   Dependents Prison_Offense Prison_Years Prior_Arrest_Episodes_Felony  \
13          0           Drug    1-2 years                   10 or more   

   Prior_Arrest_Episodes_Misd Prior_Arrest_Episodes_Violent  \
13                          3                             1   

   Prior_Arrest_Episodes_Property Prior_Arrest_Episodes_Drug  \
13                              1                  5 or more   

   Prior_Arrest_Episodes_PPViolationCharges  Prior_Arrest_Episodes_DVCharges  \
13                                5 or more                            False   

    Prior_Arrest_Episodes_GunCharges Prior_Conviction_Episodes_Felony  \
13                             False                   

In [6]:
initial_rows = len(imputed_df)
imputed_df = imputed_df.dropna(how="any")
final_rows = len(imputed_df)
rows_dropped = initial_rows - final_rows
print(f"Rows dropped: {rows_dropped}")

Rows dropped: 6419


In [7]:
print(len(imputed_df))

19416


In [8]:
imputed_df.to_csv("NIJ_clean.csv", index=False)